# 04 ML Model Comparison and Tuning
## HR Employee Attrition Analysis

This notebook follows directly from the **Baseline Model** notebook and aligns with the project README. It extends the analysis from a simple baseline logistic regression model to a comparison of multiple machine learning models and introduces basic hyperparameter tuning.


## Alignment to README

This notebook supports the README project goals by helping to:

- identify key drivers of employee attrition
- assess the impact of salary, satisfaction, and employment factors
- provide actionable retention insights through predictive modelling

It also supports the project plan by progressing from:

1. Data Collection  
2. Data Cleaning & Preparation  
3. Exploratory Data Analysis  
4. Feature Analysis  
5. **Predictive Modelling and Comparison**

The overall project framing comes from the README. fileciteturn6file0


## Link to Previous Notebooks

This notebook is designed to follow on from:

- **Data Cleaning & Preparation** notebook
- **EDA HR Attrition Distinction** notebook
- **03 Baseline Model Follows EDA** notebook

### What was established previously

From the EDA and baseline modelling stages, the following were identified as important predictors of attrition:

- Job Satisfaction
- Work-Life Balance
- Monthly Income
- Years at Company
- Department and Job Role

The baseline model provided an initial benchmark. This notebook now compares multiple algorithms to determine whether performance can be improved while still keeping results interpretable.


## Objectives

- Load the cleaned HR attrition dataset
- Prepare features and target variable
- Reproduce the baseline model for comparison
- Compare multiple classification models
- Tune the strongest candidate model
- Evaluate results using classification metrics
- Reflect on how the model results support the business problem and README narrative


---
# Import Libraries


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)


---
# Load Dataset

This notebook first attempts to load the cleaned dataset from the preparation stage. If that file is unavailable, it falls back to the raw HR Employee Attrition dataset.


In [3]:
import os
from pathlib import Path
import pandas as pd

project_root_candidates = [
    Path.cwd(),
    Path.cwd() / "Employee_attrition_analysis3",
    Path.cwd().parent / "Employee_attrition_analysis3"
]
project_root = next((path for path in project_root_candidates if (path / "Dataset").exists()), None)

if project_root is None:
    raise FileNotFoundError("Could not locate the project root containing the Dataset folder.")

os.chdir(project_root)
print(f"Working directory updated to project root: {os.getcwd()}")

preferred_paths = [
    project_root / "Dataset" / "Processed" / "hr_employee_attrition_cleaned.csv",
    project_root / "Dataset" / "Raw" / "HR Employee Attrition.csv"
]

data_path = None
for path in preferred_paths:
    if path.exists():
        data_path = path
        break

if data_path is None:
    raise FileNotFoundError(
        "Could not find the HR Employee Attrition dataset. "
        "Expected Dataset/Processed/hr_employee_attrition_cleaned.csv or Dataset/Raw/HR Employee Attrition.csv"
    )

df = pd.read_csv(data_path)
print("Loaded dataset from:", data_path)
print("Shape:", df.shape)
df.head()


Working directory updated to project root: c:\Python Projects\Employee_attrition_analysis3
Loaded dataset from: c:\Python Projects\Employee_attrition_analysis3\Dataset\Processed\hr_employee_attrition_cleaned.csv
Shape: (1470, 35)


,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EmployeeCount,EmployeeNumber,...,RelationshipSatisfaction,StandardHours,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
0,41,Yes,Travel_Rarely,1102,Sales,1,2,Life Sciences,1,1,...,1,80,0,8,0,1,6,4,0,5
1,49,No,Travel_Frequently,279,Research & Development,8,1,Life Sciences,1,2,...,4,80,1,10,3,3,10,7,1,7
2,37,Yes,Travel_Rarely,1373,Research & Development,2,2,Other,1,4,...,2,80,0,7,3,3,0,0,0,0
3,33,No,Travel_Frequently,1392,Research & Development,3,4,Life Sciences,1,5,...,3,80,0,8,3,3,8,7,3,0
4,27,No,Travel_Rarely,591,Research & Development,2,1,Medical,1,7,...,4,80,1,6,3,3,2,2,2,2


## Dataset Context

The project uses the **HR Employee Attrition dataset** described in the README as containing demographics, employment details, compensation, satisfaction metrics, and the target variable **Attrition (Yes/No)**. The README also notes that this is a static snapshot dataset, which supports pattern identification but not time-based trend analysis. fileciteturn6file0


In [ ]:
df.info()


---
# Prepare Features and Target

This modelling step follows directly from the EDA findings. Attrition is the target variable, and the remaining fields are treated as candidate predictors.


In [ ]:
target = "Attrition"

if target not in df.columns:
    raise ValueError("The dataset must contain an 'Attrition' column.")

X = df.drop(columns=[target]).copy()
y = df[target].copy()

# Convert target to binary if required
if y.dtype == "object":
    y = y.map({"Yes": 1, "No": 0})

print("Feature shape:", X.shape)
print("Target distribution:")
print(y.value_counts(dropna=False))


In [ ]:
categorical_cols = X.select_dtypes(include=["object", "category"]).columns.tolist()
numerical_cols = X.select_dtypes(include=[np.number]).columns.tolist()

print("Categorical columns:", len(categorical_cols))
print("Numerical columns:", len(numerical_cols))


## Train/Test Split

A stratified split is used so that the attrition class balance is preserved across training and test datasets.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)


---
# Preprocessing Pipeline

To keep the workflow reproducible:

- categorical variables are one-hot encoded
- numerical variables are scaled

This allows fair comparison across models.


In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numerical_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols)
    ]
)


---
# Define Models for Comparison

The baseline notebook used **Logistic Regression**. This notebook keeps it for benchmarking and compares it against additional models:

- Logistic Regression
- Decision Tree
- Random Forest
- Gradient Boosting

This helps assess whether more advanced methods improve predictive performance.


In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(random_state=42),
    "Gradient Boosting": GradientBoostingClassifier(random_state=42)
}


---
# Train and Compare Models


In [ ]:
results = []

for name, model in models.items():
    pipeline = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("model", model)
    ])

    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)

    if hasattr(pipeline, "predict_proba"):
        y_proba = pipeline.predict_proba(X_test)[:, 1]
    else:
        y_proba = None

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, zero_division=0),
        "Recall": recall_score(y_test, y_pred, zero_division=0),
        "F1 Score": f1_score(y_test, y_pred, zero_division=0),
        "ROC AUC": roc_auc_score(y_test, y_proba) if y_proba is not None else np.nan
    })

results_df = pd.DataFrame(results).sort_values(by="F1 Score", ascending=False)
results_df


**Insight:** Model comparison helps determine whether the simple baseline can be improved. In employee attrition analysis, recall and F1 score are especially useful because the business is interested in identifying employees at risk of leaving, not just overall accuracy.

**Business Recommendation:** Choose the model that best balances interpretability and detection of likely attrition cases. A slightly more complex model may be justified if it substantially improves recall for at-risk employees.


---
# Visual Comparison of Model Performance


In [ ]:
results_plot = results_df.set_index("Model")[["Accuracy", "Precision", "Recall", "F1 Score", "ROC AUC"]]
results_plot.plot(kind="bar", figsize=(12, 6))
plt.title("Model Comparison Metrics")
plt.ylabel("Score")
plt.xticks(rotation=45, ha="right")
plt.ylim(0, 1)
plt.tight_layout()
plt.show()


**Insight:** Comparing multiple metrics makes it easier to avoid selecting a model purely on accuracy. In attrition modelling, a model with better recall may be more valuable because it flags more employees who could leave.

**Business Recommendation:** Prioritise the metric that best supports HR decision-making. For example, if the organisation wants early warning signals, favour recall and F1 over raw accuracy.


---
# Select Best Model

The best model is selected using **F1 Score** as the primary comparison metric, because attrition prediction requires a balance between precision and recall.


In [ ]:
best_model_name = results_df.iloc[0]["Model"]
print("Best model based on F1 Score:", best_model_name)


---
# Hyperparameter Tuning

To refine the comparison, the strongest candidate model is tuned using GridSearchCV.

For this notebook, tuning is demonstrated on **Random Forest**. If a different model performs best in your actual dataset, you can adapt the tuning section accordingly.


In [ ]:
rf_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(random_state=42))
])

param_grid = {
    "model__n_estimators": [100, 200],
    "model__max_depth": [None, 5, 10],
    "model__min_samples_split": [2, 5]
}

grid_search = GridSearchCV(
    rf_pipeline,
    param_grid=param_grid,
    cv=5,
    scoring="f1",
    n_jobs=-1
)

grid_search.fit(X_train, y_train)

print("Best parameters:", grid_search.best_params_)
print("Best CV F1 score:", grid_search.best_score_)


**Insight:** Hyperparameter tuning improves model performance by testing different combinations of settings rather than relying on defaults.

**Business Recommendation:** Use tuning before production deployment so the model is not only functional, but appropriately optimised for identifying employees at attrition risk.


---
# Evaluate Tuned Model on Test Set


In [ ]:
best_tuned_model = grid_search.best_estimator_

y_pred_tuned = best_tuned_model.predict(X_test)
y_proba_tuned = best_tuned_model.predict_proba(X_test)[:, 1]

print("Classification Report:")
print(classification_report(y_test, y_pred_tuned))

tuned_metrics = {
    "Accuracy": accuracy_score(y_test, y_pred_tuned),
    "Precision": precision_score(y_test, y_pred_tuned, zero_division=0),
    "Recall": recall_score(y_test, y_pred_tuned, zero_division=0),
    "F1 Score": f1_score(y_test, y_pred_tuned, zero_division=0),
    "ROC AUC": roc_auc_score(y_test, y_proba_tuned)
}

pd.DataFrame([tuned_metrics])


In [ ]:
cm = confusion_matrix(y_test, y_pred_tuned)
disp = ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot(cmap="Blues")
plt.title("Confusion Matrix - Tuned Random Forest")
plt.show()


**Insight:** The confusion matrix shows where the tuned model performs well and where misclassifications still occur. This is important because false negatives represent employees who may leave but were not flagged.

**Business Recommendation:** Use model outputs as decision support rather than as a standalone decision-maker. HR teams should treat flagged cases as a prompt for further review and targeted interventions.


---
# Feature Importance

Feature importance helps connect the model back to the EDA findings and the README hypotheses.


In [ ]:
# Get feature names after preprocessing
ohe = best_tuned_model.named_steps["preprocessor"].named_transformers_["cat"]
encoded_cat_names = ohe.get_feature_names_out(categorical_cols)

feature_names = numerical_cols + list(encoded_cat_names)

importances = best_tuned_model.named_steps["model"].feature_importances_

importance_df = pd.DataFrame({
    "Feature": feature_names,
    "Importance": importances
}).sort_values(by="Importance", ascending=False).head(15)

importance_df


In [ ]:
plt.figure(figsize=(10, 6))
sns.barplot(data=importance_df, x="Importance", y="Feature")
plt.title("Top 15 Feature Importances")
plt.tight_layout()
plt.show()


**Insight:** Feature importance shows which variables are most influential in predicting attrition. This provides a useful bridge between EDA observations and predictive modelling.

**Business Recommendation:** Focus retention initiatives on the variables that consistently appear most influential, such as satisfaction, compensation, tenure, and specific organisational contexts.


---
# Reflection and Alignment to README

This notebook aligns with the README by extending the project from EDA into comparative modelling. It supports the project goals by:

- testing whether the EDA findings have predictive value
- comparing multiple algorithms rather than relying on one baseline only
- identifying the trade-off between interpretability and predictive performance
- providing evidence for future dashboard and recommendation outputs

The README notes future improvements such as adding machine learning prediction models and improving interactivity. This notebook directly supports that roadmap by providing the first model comparison and tuning stage. fileciteturn6file0


## Limitations

- This dataset is a static snapshot and cannot show how attrition risk changes over time
- Model predictions are only as good as the available features
- Some algorithms may perform better with class balancing or additional feature engineering
- A high-performing model does not prove causation; it only identifies predictive relationships


## Next Steps

Possible improvements after this notebook:

- test class imbalance handling techniques such as SMOTE or class weighting
- compare additional models such as XGBoost
- tune the best-performing model more extensively
- deploy model outputs into the dashboard as an attrition risk view
- add explainability methods such as SHAP for deeper interpretation


## Conclusion

This notebook follows directly from the baseline model and the EDA work. It aligns with the README by extending the project into **model comparison and tuning**, while keeping the focus on the central business problem: understanding and reducing employee attrition. The result is a stronger analytical foundation for future dashboarding, recommendation, and decision-support work.
